# Stock Price Prediction Using an LSTM-Based Recurrent Neural Network

## Introduction

The aim of this project is to build a machine learning model that predicts the next-day closing price of NVIDIA Corporation (NVDA) stock using historical closing prices from Yahoo Finance. The project follows a complete workflow: data collection, cleaning, preprocessing, sequence preparation, LSTM model development, evaluation, visualization, comparison with a traditional machine learning model, and critical analysis.

Stock price prediction is challenging because financial markets are noisy, non-stationary, and influenced by many external factors such as earnings announcements, economic conditions, interest rates, investor sentiment, and unexpected news. Historical price data can reveal patterns, but it cannot fully explain future market behavior.

Recurrent Neural Networks (RNNs), especially Long Short-Term Memory (LSTM) networks, are suitable for time-series prediction because they can learn patterns from ordered sequences. LSTMs use gating mechanisms to retain useful information over time and reduce the vanishing-gradient problem found in simple RNNs. This makes them appropriate for learning from historical stock-price windows.


## 1. Install and Import Libraries

The following libraries are used for downloading data, preprocessing, model building, evaluation, and visualization. If `yfinance` is not installed in the environment, the installation command below can be uncommented and run.


In [ ]:
# If needed in a new notebook environment, uncomment the next line:
# %pip install yfinance

from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

warnings.filterwarnings("ignore")

YFINANCE_CACHE_DIR = Path(".yfinance_cache")
YFINANCE_CACHE_DIR.mkdir(exist_ok=True)
yf.set_tz_cache_location(str(YFINANCE_CACHE_DIR))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

plt.style.use("seaborn-v0_8-whitegrid")


## 2. Data Collection

Historical NVDA stock data is downloaded from Yahoo Finance using the `yfinance` library. The requested project period is from `2015-01-01` to `2025-12-31`. In `yfinance`, the `end` date is exclusive, so `2026-01-01` is used to include data up to the final trading day of 2025.


In [ ]:
ticker = "NVDA"
start_date = "2015-01-01"
end_date = "2026-01-01"  # yfinance end date is exclusive

csv_path = Path("NVDA_stock_data.csv")

try:
    stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False, auto_adjust=False)
except Exception as error:
    print(f"Yahoo Finance download failed: {error}")
    stock_data = pd.DataFrame()

if stock_data.empty and csv_path.exists():
    print("Using cached NVDA_stock_data.csv because a fresh Yahoo Finance download was not available.")
    stock_data = pd.read_csv(csv_path, parse_dates=["Date"], index_col="Date")
elif stock_data.empty:
    raise RuntimeError("No data was downloaded and no cached NVDA_stock_data.csv file was found.")

# yfinance can sometimes return MultiIndex columns. Flatten them if needed.
if isinstance(stock_data.columns, pd.MultiIndex):
    stock_data.columns = [column[0] for column in stock_data.columns]

stock_data = stock_data.sort_index()
stock_data.to_csv(csv_path, index_label="Date")

print("Dataset saved as NVDA_stock_data.csv")
display(stock_data.head())
print("Dataset shape:", stock_data.shape)
stock_data.info()


## 3. Data Cleaning and Preprocessing

This project uses only the closing price for prediction. Missing values are checked and removed if present. The closing price is normalized using `MinMaxScaler` because neural networks usually train more efficiently when input values are on a similar scale.

To avoid future data leakage, the scaler is fitted only on the portion of data used to create the training sequences. The fitted scaler is then used to transform the whole closing-price series.


In [ ]:
print("Missing values before cleaning:")
print(stock_data.isna().sum())

stock_data = stock_data.dropna().sort_index()

close_prices = stock_data[["Close"]].copy()

plt.figure(figsize=(12, 5))
plt.plot(close_prices.index, close_prices["Close"], label="NVDA Close Price", color="navy")
plt.title("Original NVDA Closing Price Trend")
plt.xlabel("Date")
plt.ylabel("Closing Price (USD)")
plt.legend()
plt.show()

lookback = 60
raw_values = close_prices.values

# The final train/test split is applied after sequence creation.
# Fit the scaler only on values that belong to the training period.
total_sequences = len(raw_values) - lookback
train_size = int(total_sequences * 0.8)
scaler_train_end = train_size + lookback

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(raw_values[:scaler_train_end])
scaled_prices = scaler.transform(raw_values)

print("Cleaned closing-price data shape:", close_prices.shape)


## 4. Sequence Preparation

A lookback window of 60 trading days is used. For each sample, `X` contains the previous 60 normalized closing prices, and `y` contains the next-day normalized closing price. The input is reshaped into the format required by an LSTM: `(samples, time steps, features)`.


In [ ]:
def create_sequences(data, lookback_window):
    X, y = [], []
    for i in range(lookback_window, len(data)):
        X.append(data[i - lookback_window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)


X, y = create_sequences(scaled_prices, lookback)
X = X.reshape((X.shape[0], X.shape[1], 1))

sequence_dates = close_prices.index[lookback:]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 5. Train-Test Split

The data is split chronologically rather than randomly. This is important for time-series prediction because future observations must not be used to train a model that is evaluated on earlier observations. The first 80% of sequences are used for training, and the final 20% are used for testing.


In [ ]:
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

train_dates = sequence_dates[:train_size]
test_dates = sequence_dates[train_size:]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Training period:", train_dates.min().date(), "to", train_dates.max().date())
print("Testing period:", test_dates.min().date(), "to", test_dates.max().date())


## 6. LSTM Model Development

The LSTM model uses two LSTM layers with dropout regularization. The first LSTM layer returns sequences so that the second LSTM layer can process the full temporal output. The final dense layer produces one predicted normalized closing price.


In [ ]:
lstm_model = Sequential([
    Input(shape=(lookback, 1)),
    LSTM(units=50, return_sequences=True),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(units=1)
])

lstm_model.compile(optimizer="adam", loss="mean_squared_error")
lstm_model.summary()


## 7. Model Training

The model is trained for 30 epochs with a batch size of 32. A validation split of 10% is used from the training data to monitor how the model performs on unseen training-period samples. The time-series data is not shuffled.


In [ ]:
history = lstm_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error Loss")
plt.legend()
plt.show()


## 8. Prediction and Evaluation

The trained LSTM model is used to predict the closing prices for the test set. The predictions and actual values are inverse-transformed back to the original dollar price scale before calculating evaluation metrics.


In [ ]:
lstm_predictions_scaled = lstm_model.predict(X_test)

y_test_original = scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
lstm_predictions_original = scaler.inverse_transform(lstm_predictions_scaled).ravel()


def calculate_metrics(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2,
        "MAPE (%)": mape,
    }


lstm_metrics = calculate_metrics(y_test_original, lstm_predictions_original)

print("LSTM Evaluation Results")
for metric_name, metric_value in lstm_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")


## 9. Visualization of LSTM Results

The following plots compare actual and predicted closing prices and show prediction errors over time. A small table is also created to inspect individual examples.


In [ ]:
results_df = pd.DataFrame({
    "Date": test_dates,
    "Actual Price": y_test_original,
    "LSTM Predicted Price": lstm_predictions_original,
})
results_df["LSTM Error"] = results_df["Actual Price"] - results_df["LSTM Predicted Price"]
results_df["LSTM Absolute Error"] = results_df["LSTM Error"].abs()

plt.figure(figsize=(12, 5))
plt.plot(results_df["Date"], results_df["Actual Price"], label="Actual Price", color="black")
plt.plot(results_df["Date"], results_df["LSTM Predicted Price"], label="LSTM Predicted Price", color="green")
plt.title("Actual vs LSTM Predicted NVDA Closing Price")
plt.xlabel("Date")
plt.ylabel("Closing Price (USD)")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(results_df["Date"], results_df["LSTM Error"], label="Prediction Error", color="darkred")
plt.axhline(0, color="black", linewidth=1)
plt.title("LSTM Prediction Error Over Time")
plt.xlabel("Date")
plt.ylabel("Actual - Predicted (USD)")
plt.legend()
plt.show()

display(results_df[["Date", "Actual Price", "LSTM Predicted Price", "LSTM Error"]].head(10))


## 10. Alternative Model Comparison

A Random Forest Regressor is used as an alternative traditional machine learning model. Since Random Forest does not accept 3D sequence input, the same 60-day windows are flattened into 2D feature vectors. The model is evaluated using the same test split and the same metrics as the LSTM model.


In [ ]:
X_train_flat = X_train.reshape((X_train.shape[0], X_train.shape[1]))
X_test_flat = X_test.reshape((X_test.shape[0], X_test.shape[1]))

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf_model.fit(X_train_flat, y_train)

rf_predictions_scaled = rf_model.predict(X_test_flat).reshape(-1, 1)
rf_predictions_original = scaler.inverse_transform(rf_predictions_scaled).ravel()

rf_metrics = calculate_metrics(y_test_original, rf_predictions_original)

comparison_table = pd.DataFrame([lstm_metrics, rf_metrics], index=["LSTM", "Random Forest"])
display(comparison_table)

results_df["Random Forest Predicted Price"] = rf_predictions_original

plt.figure(figsize=(12, 5))
plt.plot(results_df["Date"], results_df["Actual Price"], label="Actual Price", color="black")
plt.plot(results_df["Date"], results_df["LSTM Predicted Price"], label="LSTM", color="green")
plt.plot(results_df["Date"], results_df["Random Forest Predicted Price"], label="Random Forest", color="royalblue", alpha=0.8)
plt.title("Actual vs Predicted Prices: LSTM and Random Forest")
plt.xlabel("Date")
plt.ylabel("Closing Price (USD)")
plt.legend()
plt.show()


## 11. Critical Analysis

### Strengths of the LSTM Model

The LSTM model is well suited to this task because it learns from sequences of past prices rather than treating each observation independently. The 60-day lookback window allows the model to consider recent price trends and temporal dependencies. LSTM gates help the model decide which information from previous days should be retained or forgotten, which is useful for time-series data.

### Limitations

The model is still limited because it only uses historical closing prices. Stock prices are influenced by earnings reports, product announcements, macroeconomic conditions, interest rates, geopolitical events, and investor sentiment. These factors are not included in the dataset, so the model cannot react directly to them.

Overfitting is also a risk because LSTM models have many parameters and stock data can be noisy. If the training loss decreases while validation loss increases, the model may be memorizing training patterns instead of generalizing. The model is also sensitive to market volatility because sudden price movements may not be predictable from recent closing prices alone.

### LSTM Compared with Random Forest

The Random Forest model uses the same historical window, but the window is flattened into independent lag features. This means Random Forest can learn nonlinear relationships between previous prices and the next price, but it does not explicitly model sequence order in the same way as an LSTM. If the LSTM has lower MAE, RMSE, and MAPE, it suggests that the sequential structure helped. If Random Forest performs similarly or better, it suggests that a simpler model may be sufficient for this dataset.

Good trend-following in a graph does not guarantee accurate real-world trading performance. A model can appear to follow the general direction of prices while still producing errors that are too large for profitable trading after transaction costs, risk, and changing market conditions are considered.


## 12. Conclusion

This project developed an LSTM-based recurrent neural network to predict the next-day closing price of NVDA stock using historical Yahoo Finance data from 2015 to 2025. The workflow included data collection, cleaning, normalization, sequence preparation, chronological train-test splitting, LSTM model training, evaluation, visualization, and comparison with a Random Forest model.

LSTM models are suitable for stock price prediction experiments because they can learn temporal patterns from sequential data. However, historical closing prices alone are not enough to fully explain future market behavior. The model should therefore be viewed as an educational forecasting model rather than a complete trading system.

Future improvements could include adding technical indicators, trading volume, macroeconomic variables, earnings information, sentiment analysis from news or social media, and experimenting with GRU or Transformer-based models.


## Source Code Link

https://github.com/BingJun69/ML_Final_Assesment.git
